
# Support Vector Machine (SVM) Classification Project

## Project Workflow
1. Dataset Loading  
2. Exploratory Data Analysis (EDA)  
3. Missing Value Handling  
4. Encoding Categorical Features  
5. Standardization  
6. Train / Validation / Test Split  
7. SVM Model Training  
8. Hyperparameter Tuning  
9. Evaluation Metrics  
10. Confusion Matrix  
11. ROC Curve  
12. Sample Predictions  
13. 2D Classification Plot  


In [ ]:

# =========================
# Import Libraries
# =========================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Load Dataset
# =========================

# Upload Titanic-Dataset.csv in Google Colab first

df = pd.read_csv('/content/Titanic-Dataset.csv')

# =========================
# Basic Dataset Information
# =========================

print("First 5 Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())

# =========================
# Exploratory Data Analysis
# =========================

# Survival Count Plot

plt.figure(figsize=(6,5))

sns.countplot(x='Survived', data=df)

plt.title("Survival Count")
plt.show()

# Gender Distribution

plt.figure(figsize=(6,5))

sns.countplot(x='Sex', data=df)

plt.title("Gender Distribution")
plt.show()

# Survival based on Gender

plt.figure(figsize=(6,5))

sns.countplot(x='Sex', hue='Survived', data=df)

plt.title("Survival Based on Gender")
plt.show()

# Age Distribution

plt.figure(figsize=(8,5))

sns.histplot(df['Age'], bins=30, kde=True)

plt.title("Age Distribution")
plt.show()

# Fare Distribution

plt.figure(figsize=(8,5))

sns.histplot(df['Fare'], bins=30, kde=True)

plt.title("Fare Distribution")
plt.show()

# Correlation Heatmap

plt.figure(figsize=(10,6))

numeric_df = df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")
plt.show()


In [ ]:

# =========================
# Machine Learning Libraries
# =========================

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# =========================
# Data Preprocessing
# =========================

# Drop unnecessary columns
df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1, inplace=True)

# Handle missing values

# Age -> median
age_imputer = SimpleImputer(strategy='median')
df['Age'] = age_imputer.fit_transform(df[['Age']])

# Embarked -> most frequent
embarked_imputer = SimpleImputer(strategy='most_frequent')
df['Embarked'] = embarked_imputer.fit_transform(df[['Embarked']]).ravel()

# =========================
# Encode categorical features
# =========================

le = LabelEncoder()

df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

# =========================
# Features and Target
# =========================

X = df.drop('Survived', axis=1)
y = df['Survived']

# =========================
# Train / Validation / Test Split
# =========================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print("Training Size:", X_train.shape)
print("Validation Size:", X_val.shape)
print("Test Size:", X_test.shape)

# =========================
# Feature Scaling
# =========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# =========================
# Hyperparameter Tuning
# =========================

param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

# =========================
# Validation Performance
# =========================

val_pred = best_model.predict(X_val)

print("\nValidation Accuracy:")
print(accuracy_score(y_val, val_pred))

# =========================
# Test Prediction
# =========================

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

# =========================
# Evaluation Metrics
# =========================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("\n===== Evaluation Metrics =====")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC Score:", auc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# =========================
# Confusion Matrix
# =========================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

# =========================
# ROC Curve
# =========================

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))

plt.plot(fpr, tpr, label='SVM ROC Curve')
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

# =========================
# Sample Predictions
# =========================

sample_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Probability': y_prob[:10]
})

print("\nSample Predictions:\n")
print(sample_df)

# =========================
# 2D Classification Plot
# =========================

from sklearn.decomposition import PCA

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_train)

svm_2d = SVC(kernel='rbf')

svm_2d.fit(X_pca, y_train)

x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, 0.02),
    np.arange(y_min, y_max, 0.02)
)

Z = svm_2d.predict(
    np.c_[xx.ravel(), yy.ravel()]
)

Z = Z.reshape(xx.shape)

plt.figure(figsize=(8,6))

plt.contourf(xx, yy, Z, alpha=0.3)

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y_train,
    edgecolors='k'
)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("2D Classification Plot")

plt.show()



# Conclusion

This project implemented a complete Support Vector Machine (SVM) classification workflow using the Titanic dataset.

The workflow included:
- Data preprocessing
- Exploratory Data Analysis
- Feature encoding
- Feature scaling
- Hyperparameter tuning
- Model evaluation

The final model performance was evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Visualization techniques such as confusion matrix, ROC curve, and 2D classification plot were also used to analyze the model performance.
